# Lọc comment tiếng Anh

Notebook này chạy độc lập sau `clearing_data.ipynb`. Đầu vào là
`comments_cleaned_to_5_4.csv` gồm hai cột `row_id`, `comment`.

Chỉ xóa comment được nhận diện là tiếng Anh với độ tin cậy cao (mặc định 0,95)
khi so sánh giữa tiếng Việt và tiếng Anh. Comment trộn Việt-Anh có độ tin cậy
thấp sẽ được giữ lại để hạn chế xóa nhầm.


In [ ]:
# Cài thư viện vào kernel hiện tại

%pip install lingua-language-detector pandas


In [ ]:
from pathlib import Path
import json

import pandas as pd
import re
from lingua import Language, LanguageDetectorBuilder


In [ ]:
# Tìm file dataset đầu vào

candidate_paths = [
    Path('../data/interim/comments_cleaned_to_5_4.csv'),
    Path('data/interim/comments_cleaned_to_5_4.csv'),
    Path('Preprocessing/data/interim/comments_cleaned_to_5_4.csv'),
]
input_path = next((path for path in candidate_paths if path.exists()), None)

if input_path is None:
    raise FileNotFoundError('Không tìm thấy comments_cleaned_to_5_4.csv.')

print(f'Đọc dữ liệu từ: {input_path.resolve()}')


In [ ]:
# Đọc và kiểm tra schema

comments = pd.read_csv(
    input_path,
    encoding='utf-8-sig',
    dtype='string',
    keep_default_na=False,
)

required_columns = {'row_id', 'comment'}
missing_columns = required_columns - set(comments.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing_columns)}')

comments['comment'] = comments['comment'].fillna('').astype('string').str.strip()
print(f'Số comment đầu vào: {len(comments):,}')


In [ ]:
# Tạo bộ nhận diện ngôn ngữ Việt-Anh

language_detector = LanguageDetectorBuilder.from_languages(
    Language.VIETNAMESE,
    Language.ENGLISH,
).build()

CONFIG_PATHS = [
    Path('../config/preprocessing_config.json'),
    Path('config/preprocessing_config.json'),
    Path('Preprocessing/config/preprocessing_config.json'),
]
CONFIG_PATH = next((path for path in CONFIG_PATHS if path.exists()), None)
if CONFIG_PATH is None:
    raise FileNotFoundError('Không tìm thấy preprocessing_config.json.')
with CONFIG_PATH.open(encoding='utf-8') as config_file:
    PREPROCESSING_CONFIG = json.load(config_file)
ENGLISH_CONFIDENCE_THRESHOLD = PREPROCESSING_CONFIG['english_confidence_threshold']
print(f'Ngưỡng nhận diện tiếng Anh: {ENGLISH_CONFIDENCE_THRESHOLD}')


In [ ]:
# Hàm nhận diện comment tiếng Anh

def is_confident_english_comment(text, minimum_confidence):
    text = str(text).strip()
    if not re.search(r'[A-Za-z]', text):
        return False

    scores = {
        item.language: item.value
        for item in language_detector.compute_language_confidence_values(text)
    }
    english_score = scores.get(Language.ENGLISH, 0.0)
    vietnamese_score = scores.get(Language.VIETNAMESE, 0.0)
    return english_score >= minimum_confidence and english_score > vietnamese_score


In [ ]:
# Đánh dấu comment tiếng Anh

english_audit = comments.copy(deep=True)
english_audit['is_english_comment'] = english_audit['comment'].map(
    lambda text: is_confident_english_comment(
        text,
        ENGLISH_CONFIDENCE_THRESHOLD,
    )
)


In [ ]:
# Tách dữ liệu giữ lại và dữ liệu tiếng Anh bị loại

comments_without_english = english_audit.loc[
    ~english_audit['is_english_comment'],
    ['row_id', 'comment'],
].copy()

removed_english_comments = english_audit.loc[
    english_audit['is_english_comment'],
    ['row_id', 'comment'],
].copy()
removed_english_comments['remove_reason'] = 'high_confidence_english'


In [ ]:
# Kiểm tra kết quả và xuất file

english_filter_summary = pd.DataFrame({
    'metric': ['before_language_filter', 'removed_english_comments', 'after_language_filter'],
    'records': [
        len(english_audit),
        len(removed_english_comments),
        len(comments_without_english),
    ],
})

output_path = input_path.parent / 'comments_cleaned_to_5_4_no_english.csv'
comments_without_english.to_csv(output_path, index=False, encoding='utf-8-sig')

display(english_filter_summary)
display(removed_english_comments.head(20))
print(f'Đã xuất file: {output_path.resolve()}')
